In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import ccxt
import ccxt.pro as ccxtpro
from btc_model.setting.setting import get_settings
from btc_model.core.util.crypto_util import CryptoUtil


/Users/Jason/work/source/03_ThorpAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
# 获取设置
setting = get_settings('cex.sandbox.stokx')

apikey = setting['apikey']
secretkey = setting['secretkey']
passphrase = setting['passphrase']


# 初始化币安交易所
params = {
    'enableRateLimit': True,
    'proxies': {
        'http': get_settings('common')['proxies'].get('http', None),                  
        'https': get_settings('common')['proxies'].get('https', None),
    },
    'apiKey': apikey,          
    'secret': secretkey,  
    'password': passphrase,     
    'options': {
        'defaultType': 'spot',  # 可选：'spot', 'margin', 'future'
    },
    #  'headers': {
    #     'x-simulated-trading': '1'
    # }
}

exchange = ccxt.okx(params)

exchange.set_sandbox_mode(True)

exchange_pro = ccxtpro.okx(params)
exchange_pro.set_sandbox_mode(True)

In [ ]:
exchanges = CryptoUtil.create_sandbox_exchanges(['binance', 'okx'], 'spot')
exchange_1 = exchanges['binance']
exchange_1.load_markets()
exchange_2 = exchanges['okx']
exchange_2.load_markets()
hedge_exchange = CryptoUtil.create_sandbox_exchanges(['okx'], 'swap')['okx']
hedge_exchange.load_markets()

In [30]:

symbol = 'BTC/USDT'
order = exchange_2.create_order(
            symbol=symbol,
            type='limit',
            side='buy',
            amount=0.01,
            price=87160,
            params={
                'tdMode': 'isolated',
                'positionSide': 'short'   # 持仓方向，'long' 或 'short'
            }
        )

In [31]:
hedge_exchange.set_leverage(4, 'BTC/USDT:USDT')
symbol = 'BTC/USDT:USDT'
order = hedge_exchange.create_order(
            symbol=symbol,
            type='limit',
            side='buy',
            amount=0.01,
            price=87160,
            params={
                'tdMode': 'isolated',  # 全仓模式，'isolated' 为逐仓模式
                'positionSide': 'long'   # 持仓方向，'long' 或 'short'
            }
        )

In [ ]:
print(order)

In [ ]:
try:
    # 获取账户余额
    balance = exchange.fetch_balance({
        'type': 'trading'
    })
    
    print("=== 模拟账户余额 ===")
    for currency in balance['total']:
        if balance['total'][currency] > 0:
            print(f"{currency}:")
            print(f"  可用: {balance['free'][currency]}")
            print(f"  冻结: {balance['used'][currency]}")
            print(f"  总额: {balance['total'][currency]}")

except Exception as e:
    print(f"错误: {str(e)}")


In [3]:

try:
    # 现货限价单
    def place_spot_limit_order(symbol, side, amount, price):
        """
        symbol: 交易对，如 'BTC/USDT'
        side: 'buy' 或 'sell'
        amount: 数量
        price: 价格
        """
        order = exchange.create_order(
            symbol=symbol,
            type='limit',
            side=side,
            amount=amount,
            price=price
        )
        return order

    # 现货市价单
    def place_spot_market_order(symbol, side, amount):
        order = exchange.create_order(
            symbol=symbol,
            type='market',
            side=side,
            amount=amount
        )
        return order

    # 永续合约限价单
    def place_swap_limit_order(symbol, side, amount, price, leverage=1):
        """
        symbol: 交易对，如 'BTC/USDT:USDT'
        side: 'buy' 或 'sell'
        amount: 合约数量
        price: 价格
        leverage: 杠杆倍数
        """
        # 设置杠杆
        exchange.set_leverage(leverage, symbol)
        
        order = exchange.create_order(
            symbol=symbol,
            type='limit',
            side=side,
            amount=amount,
            price=price,
            params={
                'tdMode': 'cross',  # 全仓模式，'isolated' 为逐仓模式
                'posSide': 'long'   # 持仓方向，'long' 或 'short'
            }
        )
        return order


    # 示例使用
    # 现货限价买入
    min_amount = 0.01  # 假设最小交易量为 0.01
    spot_order = place_spot_limit_order(
        symbol='BTC/USDT',
        side='buy',
        amount=min_amount,  # 使用最小交易量
        price=40000         # 价格 40000 USDT
    )
    print("现货限价单:", spot_order)

    # 永续合约开多
    swap_order = place_swap_limit_order(
        symbol='BTC/USDT:USDT',
        side='buy',
        amount=min_amount,  # 使用最小交易量
        price=40000,
        leverage=2          # 2倍杠杆
    )
    print("永续合约限价单:", swap_order)

except Exception as e:
    print(f"下单错误: {str(e)}")

# 查询订单状态
def check_order_status(order_id, symbol):
    try:
        order = exchange.fetch_order(order_id, symbol)
        print(f"订单状态: {order['status']}")
        return order
    except Exception as e:
        print(f"查询订单错误: {str(e)}")
        return None

# 取消订单
def cancel_order(order_id, symbol):
    try:
        result = exchange.cancel_order(order_id, symbol)
        print("订单已取消")
        return result
    except Exception as e:
        print(f"取消订单错误: {str(e)}")
        return None

下单错误: name 'exchange' is not defined


In [ ]:
# 现货市价单
def place_spot_market_order(symbol, side, amount):
    order = exchange.create_order(
        symbol=symbol,
        type='market',
        side=side,
        amount=amount
    )
    return order

# 示例使用
try:
    # 使用市价单买入
    market_order = place_spot_market_order(
        symbol='BTC/USDT',
        side='buy',
        amount=0.01  # 确保数量符合最小交易量
    )
    print("现货市价单:", market_order)

except Exception as e:
    print(f"下单错误: {str(e)}")

In [4]:
crypto_util = CryptoUtil.get_instance()

In [ ]:
crypto_util.get_funding_rate(exchange=exchange, symbol='LSK/USDT:USDT')

In [ ]:
crypto_util.get_perpetual_markets(exchange=exchange)

In [10]:
markets = exchange.load_markets()

In [6]:
perpetual_markets = {}
            
for symbol, market in markets.items():
    # 筛选永续合约
    if market.get('swap') and market.get('linear'):
        market_info = {
            'symbol': symbol,
            'base': market['base'],
            'quote': market['quote'],
            'settle': market.get('settle'),
            'leverage': {
                'max': market.get('limits', {}).get('leverage', {}).get('max'),
                'min': market.get('limits', {}).get('leverage', {}).get('min', 1)
            },
            'margin_mode': market.get('margin_modes', ['isolated', 'cross']),
            'fees': {
                'maker': market.get('maker'),
                'taker': market.get('taker'),
            },
            'maintenance_margin': market.get('maintenance_margin_rate'),
            'initial_margin': market.get('initial_margin_rate'),
            'contract_size': market.get('contractSize', 1),
            'precision': {
                'price': market['precision']['price'],
                'amount': market['precision']['amount']
            }
        }
        perpetual_markets[symbol] = market_info

In [ ]:
crypto_util.get_withdrawal_fees(exchange=exchange, currencies=['BTC', 'LSK'])

In [ ]:
exchange.fetch_balance()

In [ ]:
exchange.fetch_order_book('LSK/USDT')

In [ ]:
import asyncio

await asyncio.wait_for(exchange_pro.load_markets(), 10)

In [ ]:
import asyncio
symbol = 'BTC/USDT:USDT'
await asyncio.wait_for(exchange_pro.watch_ticker(symbol=symbol), 10)

In [13]:
import ccxt.pro as ccxtpro
from btc_model.setting.setting import get_settings

# 获取设置
setting = get_settings('cex.okx')


apikey = setting['apikey']
secretkey = setting['secretkey']
passphrase = setting['passphrase']


# 初始化币安交易所
params = {
    'enableRateLimit': True,
    'proxies': {
        'http': get_settings('common')['proxies']['http'],                  
        'https': get_settings('common')['proxies']['http'],
    },
    'apiKey': apikey,          
    'secret': secretkey,    
    'password': passphrase,
    'options': {
        'defaultType': 'swap',  # 可选：'spot', 'margin', 'future'
    },
    'aiohttp_proxy': get_settings('common')['proxies']['http'],
    'ws_proxy': get_settings('common')['proxies']['http']
}

# 创建交易所实例 - 注意这里是 binance 而不是 binane
exchange_pro = ccxtpro.okx(params)

In [ ]:
await asyncio.wait_for(exchange_pro.load_markets(), 10)

In [ ]:
symbol = 'BTC/USDT:USDT'
await asyncio.wait_for(exchange_pro.watch_ticker(symbol=symbol), 10)

In [ ]:
from btc_model.core.util.crypto_util import CryptoUtil

result = CryptoUtil.get_perpetual_markets(exchange=exchange)


In [ ]:
import pandas as pd
pd.DataFrame(result).transpose()[['symbol', 'base', 'quote']].reset_index(drop=True)

In [ ]:
order = exchange.fetch_order('2338197618470871040', 'LSK/USDT')
print(order)


In [60]:
# 创建限价单
symbol_id = 'LSK/USDT'
volume = 10
spot_price_1 = 0.5352
spot_client_id_1 = '123456789012311111111'
spot_order_1 = exchange.create_limit_buy_order(
    symbol_id, 
    volume,
    spot_price_1,
    params={
        'clientOrderId': spot_client_id_1
    }
)


In [57]:
len(spot_client_id_1)

32

In [42]:
order = exchange.fetch_order(id=None, symbol=symbol_id, params={
    'clientOrderId': spot_client_id_1
})

print(order)

{'info': {'accFillSz': '0', 'algoClOrdId': '', 'algoId': '', 'attachAlgoClOrdId': '', 'attachAlgoOrds': [], 'avgPx': '', 'cTime': '1742346931067', 'cancelSource': '', 'cancelSourceReason': '', 'category': 'normal', 'ccy': '', 'clOrdId': '15', 'fee': '0', 'feeCcy': 'LSK', 'fillPx': '', 'fillSz': '0', 'fillTime': '', 'instId': 'LSK-USDT', 'instType': 'SPOT', 'isTpLimit': 'false', 'lever': '', 'linkedAlgoOrd': {'algoId': ''}, 'ordId': '2343593568286724096', 'ordType': 'limit', 'pnl': '0', 'posSide': 'net', 'px': '0.5352', 'pxType': '', 'pxUsd': '', 'pxVol': '', 'quickMgnType': '', 'rebate': '0', 'rebateCcy': 'USDT', 'reduceOnly': 'false', 'side': 'buy', 'slOrdPx': '', 'slTriggerPx': '', 'slTriggerPxType': '', 'source': '', 'state': 'live', 'stpId': '', 'stpMode': 'cancel_maker', 'sz': '10', 'tag': '', 'tdMode': 'cash', 'tgtCcy': '', 'tpOrdPx': '', 'tpTriggerPx': '', 'tpTriggerPxType': '', 'tradeId': '', 'uTime': '1742346931067'}, 'id': '2343593568286724096', 'clientOrderId': '15', 'timest